# Live RF Model Evaluation Against Current Market Data

This notebook evaluates the Random Forest volatility model against the most recent available market data.

There are two parts:
1. Compare archived live prediction runs across target dates.
2. Evaluate a completed 20-trading-day window where actual future volatility is now known.

## Important: predictions for today/tomorrow cannot be fully judged until 20 future trading days have passed.


# Imports

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Paths

In [ ]:
PROJECT_ROOT = Path.cwd()

while not (PROJECT_ROOT / "data").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

RF_MODELING_PATH = PROJECT_ROOT / "data" / "processed" / "modeling" / "random_forest"
LIVE_PREDICTIONS_PATH = RF_MODELING_PATH / "live_predictions"
LIVE_EVALUATION_PATH = RF_MODELING_PATH / "live_evaluation"

prediction_log_path = LIVE_PREDICTIONS_PATH / "prediction_log.csv"
evaluation_path = LIVE_EVALUATION_PATH / "latest_20d_rf_evaluation.csv"
summary_path = LIVE_EVALUATION_PATH / "latest_20d_rf_evaluation.summary.csv"

prediction_log_path, evaluation_path, summary_path

# Load Archived Live Predictions

The prediction log stores each daily prediction run in long format:
- `Date`: target prediction date
- `ticker`: asset symbol
- `predicted_future_volatility_20d`: model forecast for future 20-day volatility.

In [6]:
prediction_log = pd.read_csv(prediction_log_path, parse_dates=['Date'])

In [7]:
prediction_log.head()

,Date,ticker,predicted_future_volatility_20d
0,2026-08-05,AAPL,0.018918
1,2026-08-05,AGG,0.002464
2,2026-08-05,AMZN,0.018553
3,2026-08-05,CAT,0.017330
4,2026-08-05,GLD,0.014319


# Data Summary

In [10]:
prediction_log_summary = (
    prediction_log
    .groupby("Date")
    .agg(
        tickers=("ticker", "nunique"),
        mean_predicted_volatility=("predicted_future_volatility_20d", "mean"),
        min_predicted_volatility=("predicted_future_volatility_20d", "min"),
        max_predicted_volatility=("predicted_future_volatility_20d", "max"),
    )
    .reset_index()
)

In [13]:
prediction_log_summary

,Date,tickers,mean_predicted_volatility,min_predicted_volatility,max_predicted_volatility
0,2026-08-05,21,0.014474,0.002464,0.018918
1,2026-08-06,21,0.014178,0.002270,0.018328


# Compare Prediction Runs Across Days

This chart shows how the model's predicted volatility changed between archived target dates.

In [14]:
fig = px.line(
    prediction_log,
    x="Date",
    y="predicted_future_volatility_20d",
    color="ticker",
    markers=True,
    title="Predicted 20-Day Volatility by Ticker Across Live Runs",
)

fig.update_layout(
    xaxis_title="Prediction Target Date",
    yaxis_title="Predicted Future 20-Day Volatility",
    legend_title="Ticker",
)

fig.show()

In [15]:
latest_two_dates = sorted(prediction_log["Date"].unique())[-2:]

prediction_change = (
    prediction_log[prediction_log["Date"].isin(latest_two_dates)]
    .pivot(index="ticker", columns="Date", values="predicted_future_volatility_20d")
)

prediction_change["change"] = prediction_change.iloc[:, -1] - prediction_change.iloc[:, -2]
prediction_change["absolute_change"] = prediction_change["change"].abs()

prediction_change = prediction_change.sort_values("absolute_change", ascending=False)

prediction_change.head(10)

Date,2026-08-05 00:00:00,2026-08-06 00:00:00,change,absolute_change
ticker,,,,
GLD,0.014319,0.016161,0.001842,0.001842
VNQ,0.009939,0.008357,-0.001582,0.001582
PG,0.015553,0.014053,-0.001500,0.001500
AAPL,0.018918,0.017746,-0.001172,0.001172
JPM,0.016056,0.015071,-0.000985,0.000985
NEE,0.011503,0.012203,0.000700,0.000700
MSFT,0.018421,0.017866,-0.000556,0.000556
LMT,0.016416,0.015908,-0.000509,0.000509
SPY,0.011617,0.011183,-0.000434,0.000434


In [16]:
fig = px.bar(
    prediction_change.reset_index(),
    x="ticker",
    y="change",
    title="Change in Predicted Volatility Between Latest Two Runs",
)

fig.update_layout(
    xaxis_title="Ticker",
    yaxis_title="Prediction Change",
)

fig.show()

# Load Completed 20-Day Evaluation

This file evaluates a historical prediction date where the next 20 trading days have already happened.

That lets us compare:

`predicted_future_volatility_20d` vs. `actual_future_volatility_20d`

In [17]:
evaluation = pd.read_csv(
    evaluation_path,
    parse_dates=["Date", "future_window_start", "future_window_end"],
)

In [18]:
summary = pd.read_csv(summary_path)

In [19]:
summary

,evaluation_feature_date,horizon_trading_days,tickers,latest_market_date_in_snapshot,mean_future_window_start,mean_future_window_end,MAE,RMSE,R2
0,2026-07-08,20,21,2026-08-05,2026-07-09,2026-08-05,0.00499,0.008092,0.296525


In [20]:
evaluation.head()

,Date,ticker,future_window_start,future_window_end,predicted_future_volatility_20d,actual_future_volatility_20d,error,absolute_error,squared_error
0,2026-07-08,AAPL,2026-07-09,2026-08-05,0.016527,0.023612,0.007086,0.007086,5.020684e-05
1,2026-07-08,AGG,2026-07-09,2026-08-05,0.002341,0.002350,0.000009,0.000009,7.556968e-11
2,2026-07-08,AMZN,2026-07-09,2026-08-05,0.016072,0.040908,0.024836,0.024836,6.168168e-04
3,2026-07-08,CAT,2026-07-09,2026-08-05,0.024925,0.028476,0.003551,0.003551,1.261034e-05
4,2026-07-08,GLD,2026-07-09,2026-08-05,0.016260,0.015788,-0.000471,0.000471,2.221737e-07


In [21]:
mae = mean_absolute_error(
    evaluation["actual_future_volatility_20d"],
    evaluation["predicted_future_volatility_20d"],
)

rmse = np.sqrt(
    mean_squared_error(
        evaluation["actual_future_volatility_20d"],
        evaluation["predicted_future_volatility_20d"],
    )
)

r2 = r2_score(
    evaluation["actual_future_volatility_20d"],
    evaluation["predicted_future_volatility_20d"],
)

pd.DataFrame(
    {
        "metric": ["MAE", "RMSE", "R2"],
        "value": [mae, rmse, r2],
    }
)

,metric,value
0,MAE,0.004990
1,RMSE,0.008092
2,R2,0.296525


# Predicted vs Actual Volatility

A perfect model would put every point on the diagonal line.

In [22]:
max_vol = max(
    evaluation["actual_future_volatility_20d"].max(),
    evaluation["predicted_future_volatility_20d"].max(),
)

fig = px.scatter(
    evaluation,
    x="predicted_future_volatility_20d",
    y="actual_future_volatility_20d",
    text="ticker",
    title="Predicted vs Actual Future 20-Day Volatility",
)

fig.add_trace(
    go.Scatter(
        x=[0, max_vol],
        y=[0, max_vol],
        mode="lines",
        name="Perfect Prediction",
        line=dict(dash="dash"),
    )
)

fig.update_traces(textposition="top center")

fig.update_layout(
    xaxis_title="Predicted Future 20-Day Volatility",
    yaxis_title="Actual Future 20-Day Volatility",
)

fig.show()

# Per-Ticker Prediction Error

This shows where the model overpredicted or underpredicted volatility.

In [23]:
evaluation_sorted = evaluation.sort_values("error")

fig = px.bar(
    evaluation_sorted,
    x="ticker",
    y="error",
    title="Prediction Error by Ticker",
)

fig.update_layout(
    xaxis_title="Ticker",
    yaxis_title="Actual - Predicted Volatility",
)

fig.show()

In [24]:
fig = px.bar(
    evaluation.sort_values("absolute_error", ascending=False),
    x="ticker",
    y="absolute_error",
    title="Absolute Prediction Error by Ticker",
)

fig.update_layout(
    xaxis_title="Ticker",
    yaxis_title="Absolute Error",
)

fig.show()

# Predicted and Actual Volatility Side by Side

In [25]:
comparison_long = evaluation.melt(
    id_vars=["ticker"],
    value_vars=[
        "predicted_future_volatility_20d",
        "actual_future_volatility_20d",
    ],
    var_name="volatility_type",
    value_name="volatility",
)

comparison_long["volatility_type"] = comparison_long["volatility_type"].replace(
    {
        "predicted_future_volatility_20d": "Predicted",
        "actual_future_volatility_20d": "Actual",
    }
)

fig = px.bar(
    comparison_long,
    x="ticker",
    y="volatility",
    color="volatility_type",
    barmode="group",
    title="Predicted vs Actual Future 20-Day Volatility by Ticker",
)

fig.update_layout(
    xaxis_title="Ticker",
    yaxis_title="20-Day Volatility",
    legend_title="Type",
)

fig.show()

# Biggest Misses

These are the tickers where the model was furthest from realized volatility.

In [26]:
biggest_misses = (
    evaluation
    .sort_values("absolute_error", ascending=False)
    [
        [
            "ticker",
            "predicted_future_volatility_20d",
            "actual_future_volatility_20d",
            "error",
            "absolute_error",
        ]
    ]
)

In [27]:
biggest_misses.head(10)

,ticker,predicted_future_volatility_20d,actual_future_volatility_20d,error,absolute_error
2,AMZN,0.016072,0.040908,0.024836,0.024836
9,MSFT,0.016848,0.038420,0.021572,0.021572
8,LMT,0.017069,0.026223,0.009154,0.009154
0,AAPL,0.016527,0.023612,0.007086,0.007086
17,VNQ,0.013961,0.008201,-0.005760,0.005760
10,NEE,0.013755,0.008948,-0.004807,0.004807
16,V,0.016291,0.011979,-0.004312,0.004312
11,PG,0.016354,0.012501,-0.003854,0.003854
7,LLY,0.016435,0.020063,0.003628,0.003628
3,CAT,0.024925,0.028476,0.003551,0.003551


# Conclusion

This notebook extended the Random Forest volatility model from historical test-set evaluation to a more realistic live-data setting. First, we archived and compared two live prediction runs for the target dates 2026-08-05 and 2026-08-06. These predictions cannot be fully judged yet because the model is forecasting future 20-day volatility, meaning the actual outcome is only known after 20 trading days have passed.

To evaluate the model against real completed market data, we used the latest available date with a full 20-trading-day future window. The evaluation used feature rows from 2026-07-08 and compared the model’s predictions against actual realized volatility from 2026-07-09 through 2026-08-05. Across 21 tickers, the model achieved an MAE of about 0.00499, RMSE of about 0.00809, and R² of about 0.297.

Overall, the Random Forest showed useful predictive signal, but it was not perfect. It performed well for some assets such as AGG, XOM, QQQ, and UNH, where predicted and actual volatility were close. The largest misses were AMZN and MSFT, where actual volatility was much higher than predicted. This suggests the model can capture general volatility patterns, but it may underreact when individual stocks experience sudden volatility spikes.

The main takeaway is that the model is directionally useful for estimating future risk, especially as part of a portfolio analysis workflow, but it should not be treated as an exact forecast or trading signal. Future improvements could include collecting more daily live prediction runs, comparing against a rolling-volatility baseline in this live-evaluation notebook, retraining the model with newer data, and adding event-based features such as earnings dates or market news.